# TabPFN vs. gradient boosting: an honest benchmark

**Built with PriorLabs-TabPFN** · Author: Jeremy Chan · To run it yourself: open in Google Colab, then Runtime → Change runtime type → **T4 GPU** → Run all (about 15 minutes)

This notebook starts from Prior Labs' own [benchmarking example](https://github.com/PriorLabs/TabPFN/blob/main/examples/benchmarking_tabpfn.py), which compares TabPFN and XGBoost on German credit data by ROC-AUC and run time. It reproduces that example, then asks the questions a practitioner would ask before trusting the result:

| Part | Question |
|---|---|
| **A. German credit** | Does the result hold up once you add uncertainty, a simple baseline and repeated cross-validation? |
| **B1. Leakage** | What happens if you include a feature that wouldn't be known at prediction time? |
| **B2. Splits** | How do results change when you test on the *future* rather than on a random sample? |
| **B3. Imbalance** | Which metrics still mean something when only about 11% of examples are positive? |
| **B4. Cost** | How do fit and prediction time grow with data size? |
| **B5. Uncertainty** | Are the probabilities calibrated, and how much do results vary with the random seed? |

**This is not a claim that TabPFN always wins.** The aim is to show *where* it helps and where it doesn't.

### Headline results (reference run: Colab T4 GPU, full tables in `results/summary.md`)
* **German credit (1,000 rows):** TabPFN v2 with no tuning had the highest mean ROC-AUC over 15 CV folds (0.801 ± 0.031). But it was only +0.004 ahead of default XGBoost (won 9 of 15 folds) and +0.012 ahead of logistic regression (12 of 15). That margin is inside the fold-to-fold noise.
* **Bank marketing, random split (33k training rows):** XGBoost was significantly better, 0.816 vs 0.806 (ΔAUC −0.010, 95% CI −0.017 to −0.003). TabPFN v2 reads at most 10k rows; trained on the *same* 10k rows, XGBoost tied it.
* **Bank marketing, temporal split (test on 2010):** TabPFN degraded least. ROC-AUC was 0.708 vs 0.674 for XGBoost (ΔAUC +0.033, CI +0.013 to +0.054), and its calibration error was 0.089 vs 0.217.
* **Small data:** with 250 training rows TabPFN scored 0.761 vs 0.725 for logistic regression and 0.705 for XGBoost. By about 1,000 rows logistic regression had caught up.
* **Cost:** with 10k rows of context, TabPFN took 23 s to score 2,000 rows on a T4 (11.6 ms per row), against about 0.01 ms per row for XGBoost. That's roughly 1,000× slower prediction.

### In plain English: what is TabPFN?
Most machine-learning models start from scratch: you give them a table of past examples and they *train* on it. TabPFN is different. It is a neural network that has already been trained on millions of **synthetic** tables, so it has learned a general sense of how columns relate to an outcome. When you call `fit`, it doesn't retrain. It reads your training rows as **context**, much like a language model reads a prompt, and makes predictions in a single forward pass. That makes it fast and strong on small tables. It also means prediction cost grows with how much context you give it, and there is a cap on how many rows it can take.

### Model licence (read before reusing)
* The `tabpfn` library code is Apache-2.0.
* The library's **default** weights (TabPFN-2.5 and later, including the current default) are under **non-commercial** licences.
* The TabPFN-2 weights don't require a login; the later versions need you to accept a licence through Prior Labs first.
* This notebook therefore **explicitly selects the TabPFN-2 weights**, which use the *Prior Labs License* (Apache 2.0 with an additional attribution requirement, hence the "Built with PriorLabs-TabPFN" line above). Check the current licence on the model card before any commercial use.
* Datasets: UCI Statlog German Credit and UCI Bank Marketing (Moro et al., 2014), both public research datasets.

*Citation: Hollmann et al., "Accurate predictions on small data with a tabular foundation model", Nature 637 (2025).*

In [ ]:
# Setup (Colab). Versions are pinned so results can be reproduced.
%pip -q install "tabpfn==10.2.0" "xgboost==3.4.1" "lightgbm==4.7.0" tabulate

In [ ]:
import os, io, time, json, zipfile, urllib.request, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
import sklearn, xgboost as xgb, lightgbm as lgb, tabpfn
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss, brier_score_loss
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve
from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion
warnings.filterwarnings("ignore")

SEED = 42
HAS_GPU = torch.cuda.is_available()
FAST = os.environ.get("FAST", "0") == "1"            # quick-run switch, for testing only
MAX_CONTEXT = 1_000 if FAST else (10_000 if HAS_GPU else 3_000)   # TabPFN v2 was pre-trained for ≤10k rows
SMOKE_CKPT = os.environ.get("TABPFN_SMOKE_CKPT")     # internal code test with random weights. Leave unset.
OUT = "results"; os.makedirs(OUT, exist_ok=True)

print(f"tabpfn {tabpfn.__version__} | xgboost {xgb.__version__} | lightgbm {lgb.__version__} | sklearn {sklearn.__version__} | torch {torch.__version__}")
print("Device:", torch.cuda.get_device_name(0) if HAS_GPU else "CPU only (TabPFN will be slow; context capped at %d rows)" % MAX_CONTEXT)
if SMOKE_CKPT:
    print("\n*** SMOKE TEST: TabPFN is using RANDOM weights. TabPFN numbers are meaningless. Do not publish. ***\n")

## Shared helpers
Every model goes through the same interface: `runner(X_train, y_train, X_test) -> (probabilities, fit_seconds, predict_seconds)`. That way each model gets the same data, the same metrics and the same timing method.

**Baselines included, from weakest to strongest:**
1. **Prior**: always predicts the training base rate. Any useful model must beat this.
2. **Logistic regression**: a linear model with one-hot encoded categories. Simple, fast and often hard to beat on small, clean data.
3. **XGBoost, sensible defaults**: exactly the parameters from Prior Labs' example.
4. **XGBoost, CV-tuned rounds**: number of trees chosen by 5-fold CV with early stopping, *using the training fold only*.
5. **LightGBM**: library defaults.
6. **TabPFN v2**: library defaults, no tuning.

Timing note: the TabPFN weights download and the GPU initialise on the first call, so we run one warm-up fit and leave it out of all timings.

In [ ]:
def as_model_frame(X):
    # Non-numeric columns become pandas 'category' (understood by XGBoost, LightGBM and TabPFN).
    X = X.copy()
    for c in X.columns:
        if not pd.api.types.is_numeric_dtype(X[c]):
            X[c] = X[c].astype(str).astype("category")
    return X

def _cats(X): return [c for c in X.columns if isinstance(X[c].dtype, pd.CategoricalDtype)]

def run_prior(Xtr, ytr, Xte):
    t = time.perf_counter(); p = np.full(len(Xte), float(np.mean(ytr))); return p, 0.0, time.perf_counter() - t

def run_logreg(Xtr, ytr, Xte):
    cat = _cats(Xtr); num = [c for c in Xtr.columns if c not in cat]
    pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat), ("num", StandardScaler(), num)])
    m = make_pipeline(pre, LogisticRegression(max_iter=5000))
    t = time.perf_counter(); m.fit(Xtr, ytr); f = time.perf_counter() - t
    t = time.perf_counter(); p = m.predict_proba(Xte)[:, 1]; return p, f, time.perf_counter() - t

XGB_PARAMS = {  # identical to Prior Labs' benchmarking example
    "learning_rate": 0.05, "max_depth": 6, "min_child_weight": 1, "subsample": 0.9,
    "colsample_bytree": 0.9, "reg_lambda": 1.0, "tree_method": "hist",
    "eval_metric": "auc", "objective": "binary:logistic", "seed": SEED,
}

def run_xgb(Xtr, ytr, Xte, rounds=100, seed=SEED):
    params = {**XGB_PARAMS, "seed": seed}
    t = time.perf_counter()
    dtr = xgb.DMatrix(Xtr, label=ytr, enable_categorical=True)
    b = xgb.train(params, dtr, num_boost_round=rounds); f = time.perf_counter() - t
    t = time.perf_counter(); p = b.predict(xgb.DMatrix(Xte, enable_categorical=True)); return p, f, time.perf_counter() - t

def run_xgb_cv(Xtr, ytr, Xte):
    # Rounds chosen by 5-fold CV on the TRAINING data only, then refit. Tuning time counts as fit time.
    t = time.perf_counter()
    dtr = xgb.DMatrix(Xtr, label=ytr, enable_categorical=True)
    cv = xgb.cv(XGB_PARAMS, dtr, num_boost_round=1000, nfold=5, stratified=True, early_stopping_rounds=20, seed=SEED)
    b = xgb.train(XGB_PARAMS, dtr, num_boost_round=len(cv)); f = time.perf_counter() - t
    t = time.perf_counter(); p = b.predict(xgb.DMatrix(Xte, enable_categorical=True)); return p, f, time.perf_counter() - t

def run_lgbm(Xtr, ytr, Xte):
    m = lgb.LGBMClassifier(random_state=SEED, verbose=-1)
    t = time.perf_counter(); m.fit(Xtr, ytr); f = time.perf_counter() - t
    t = time.perf_counter(); p = m.predict_proba(Xte)[:, 1]; return p, f, time.perf_counter() - t

def make_tabpfn(seed=0):
    if SMOKE_CKPT:
        return TabPFNClassifier(model_path=SMOKE_CKPT, device="cpu", n_estimators=2, random_state=seed)
    return TabPFNClassifier.create_default_for_version(ModelVersion.V2, device="auto", random_state=seed)

def context_sample(Xtr, ytr, n, seed=SEED, recent=False):
    # TabPFN reads the training rows as context, capped at n. Stratified random sample, or the most recent n rows.
    if len(Xtr) <= n: return Xtr, ytr
    if recent: return Xtr.iloc[-n:], ytr.iloc[-n:]
    Xs, _, ys, _ = train_test_split(Xtr, ytr, train_size=n, stratify=ytr, random_state=seed)
    return Xs, ys

def run_tabpfn(Xtr, ytr, Xte, seed=0, max_context=None, recent=False):
    Xc, yc = context_sample(Xtr, ytr, max_context or MAX_CONTEXT, seed=SEED + seed, recent=recent)
    m = make_tabpfn(seed)
    t = time.perf_counter(); m.fit(Xc, yc); f = time.perf_counter() - t
    t = time.perf_counter(); p = m.predict_proba(Xte)[:, 1]
    if HAS_GPU: torch.cuda.synchronize()
    return p, f, time.perf_counter() - t

def metrics(y, p):
    y = np.asarray(y); p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return {"ROC-AUC": roc_auc_score(y, p), "PR-AUC": average_precision_score(y, p),
            "Log loss": log_loss(y, p), "Brier": brier_score_loss(y, p),
            "Mean pred": p.mean(), "Base rate": y.mean()}

def run_all(models, Xtr, ytr, Xte, yte, tag=""):
    rows, preds = [], {}
    for name, fn in models.items():
        p, f, pr = fn(Xtr, ytr, Xte); preds[name] = p
        rows.append({"Setting": tag, "Model": name, **metrics(yte, p), "Fit s": f, "Predict s": pr})
    return pd.DataFrame(rows), preds

def boot_ci(y, p, fn=roc_auc_score, B=1000, seed=0):
    rng = np.random.default_rng(seed); y = np.asarray(y); p = np.asarray(p); s = []
    for _ in range(B):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() != y[i].max(): s.append(fn(y[i], p[i]))
    return np.percentile(s, [2.5, 97.5])

def paired_boot_diff(y, p1, p2, fn=roc_auc_score, B=1000, seed=0):
    # Bootstrap of metric(p1) - metric(p2) on the same resampled rows (paired comparison).
    rng = np.random.default_rng(seed); y = np.asarray(y); d = []
    for _ in range(B):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() != y[i].max(): d.append(fn(y[i], p1[i]) - fn(y[i], p2[i]))
    d = np.array(d); return d.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5), (d > 0).mean()

def show(df, cols=("ROC-AUC", "PR-AUC", "Log loss", "Brier", "Mean pred", "Base rate", "Fit s", "Predict s")):
    keep = [c for c in df.columns if c not in cols] + [c for c in cols if c in df.columns]
    return df[keep].round(4)

SUMMARY = []   # (title, DataFrame) pairs written to results/summary.md at the end

---
# Part A: German credit (reproduce, then stress-test)
**Data:** 1,000 loan applicants, 20 features, outcome = good or bad credit (30% bad). This is the dataset in Prior Labs' example (OpenML ID 46562). If OpenML can't be reached, we load a public mirror of the same UCI data.

We predict **bad credit = 1**. ROC-AUC is the same whichever class you call positive, so this doesn't change the comparison with Prior Labs' example.

In [ ]:
def load_german_credit():
    try:
        from sklearn.datasets import fetch_openml
        X, y = fetch_openml(data_id=46562, as_frame=True, return_X_y=True)
        src = "OpenML 46562 (as in Prior Labs' example)"
    except Exception as e:
        url = "https://raw.githubusercontent.com/selva86/datasets/master/GermanCredit.csv"
        d = pd.read_csv(url); y = d.pop("credit_risk"); X = d
        src = f"mirror of UCI Statlog German Credit ({url})"
    y = pd.Series(pd.to_numeric(pd.Series(y).astype(str), errors="coerce"), index=X.index)
    y_bad = (y == y.value_counts().idxmin()).astype(int)            # minority class = bad credit
    assert abs(y_bad.mean() - 0.30) < 0.01, "unexpected class balance"
    return as_model_frame(X), y_bad, src

Xg, yg, src_g = load_german_credit()
print("Source:", src_g, "| shape", Xg.shape, "| bad-credit rate %.2f" % yg.mean(), "| categorical cols:", len(_cats(Xg)))

# Warm-up: downloads the weights and initialises the device, kept out of all timings
_ = run_tabpfn(Xg.iloc[:100], yg.iloc[:100], Xg.iloc[100:120])
print("TabPFN warm-up done")

### A1. Reproduction: same split as Prior Labs (80/20, stratified, `random_state=42`)

In [ ]:
MODELS = {
    "Prior (base rate)": run_prior,
    "Logistic regression": run_logreg,
    "XGBoost (sensible defaults)": run_xgb,
    "XGBoost (CV-tuned rounds)": run_xgb_cv,
    "LightGBM (defaults)": run_lgbm,
    "TabPFN v2 (defaults)": run_tabpfn,
}
Xtr, Xte, ytr, yte = train_test_split(Xg, yg, test_size=0.2, stratify=yg, random_state=42)
a1, a1_preds = run_all(MODELS, Xtr, ytr, Xte, yte, tag="German credit, single split")
SUMMARY.append(("A1. German credit: single 80/20 split (as in Prior Labs' example)", show(a1)))
show(a1)

### A2. How much should we trust that one split?
The test set has only **200 rows (60 bad loans)**. The confidence intervals below show how far ROC-AUC could move if we drew a different 200 rows. If two models' intervals overlap heavily, this split **cannot tell them apart**.

In [ ]:
rows = []
for name in ["Logistic regression", "XGBoost (sensible defaults)", "XGBoost (CV-tuned rounds)", "LightGBM (defaults)", "TabPFN v2 (defaults)"]:
    lo, hi = boot_ci(yte, a1_preds[name]); rows.append({"Model": name, "ROC-AUC": roc_auc_score(yte, a1_preds[name]), "95% CI low": lo, "95% CI high": hi})
a2 = pd.DataFrame(rows)
for other in ["XGBoost (CV-tuned rounds)", "Logistic regression"]:
    m, lo, hi, share = paired_boot_diff(yte, a1_preds["TabPFN v2 (defaults)"], a1_preds[other])
    print(f"TabPFN minus {other}: ΔAUC = {m:+.3f}  (95% CI {lo:+.3f} to {hi:+.3f}; TabPFN ahead in {share:.0%} of resamples)")
SUMMARY.append(("A2. Bootstrap 95% CIs on the 200-row test set", a2.round(3)))
a2.round(3)

### A3. Repeated stratified cross-validation (5 folds × 3 repeats = 15 test sets)
A fairer comparison: every model sees the same 15 train/test splits, and we compare them **split by split** (paired). XGBoost's CV tuning runs *inside* each training fold, so no information from the test fold leaks into it.

In [ ]:
cv_models = {k: v for k, v in MODELS.items() if k != "Prior (base rate)"}
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=1 if FAST else 3, random_state=SEED)
fold_rows = []
for k, (tr, te) in enumerate(rskf.split(Xg, yg)):
    for name, fn in cv_models.items():
        p, f, pr = fn(Xg.iloc[tr], yg.iloc[tr], Xg.iloc[te])
        fold_rows.append({"fold": k, "Model": name, "ROC-AUC": roc_auc_score(yg.iloc[te], p),
                          "Log loss": log_loss(yg.iloc[te], np.clip(p, 1e-6, 1 - 1e-6)), "Fit+predict s": f + pr})
folds = pd.DataFrame(fold_rows)
a3 = folds.groupby("Model").agg(AUC_mean=("ROC-AUC", "mean"), AUC_sd=("ROC-AUC", "std"),
                                LogLoss_mean=("Log loss", "mean"), Seconds_mean=("Fit+predict s", "mean"))
wide = folds.pivot(index="fold", columns="Model", values="ROC-AUC")
ref = wide["TabPFN v2 (defaults)"]
a3["TabPFN minus model (mean ΔAUC)"] = [(ref - wide[m]).mean() for m in a3.index]
a3["TabPFN wins (folds)"] = [f"{int((ref > wide[m]).sum())}/{len(ref)}" for m in a3.index]
a3 = a3.sort_values("AUC_mean", ascending=False)
SUMMARY.append(("A3. Repeated stratified 5-fold CV (paired)", a3.round(4)))
a3.round(4)

*Note: the TabPFN row in A3 is compared with itself, so ignore its "wins" column.*

**How to read Part A.** Ask three things. (1) Is TabPFN's lead over XGBoost bigger than the confidence interval from A2? (2) Does it beat **logistic regression**, which is often the real competitor on 1,000-row credit data? (3) Does the lead hold in most of the 15 folds, or only on average? A typical honest conclusion is something like *"best on average, by a margin about the size of the fold-to-fold noise"*. That is useful (no tuning needed), but it is not a decisive win.

---
# Part B: Bank marketing (leakage, time, imbalance, cost, uncertainty)
**Data:** 41,188 phone calls from a Portuguese bank's term-deposit campaigns, May 2008 to November 2010 (Moro et al., 2014). Outcome = did the customer subscribe (about 11% yes). This dataset is good for teaching because it contains a well-known **leaky feature** and a strong **shift over time**.

### Reconstructing the timeline
The original UCI file is in date order but has no year column, and some copies are shuffled. Each calendar month can still be recovered without guessing. `cons.price.idx` and `cons.conf.idx` are **monthly** economic indicators, so each (month, CPI, confidence) combination is exactly one calendar month: 26 in total. `nr.employed` is **quarterly**, and its values fall into three groups, one per year. The `euribor3m` column lets you check this against the public 3-month Euribor rate: for example, March 2009 should be around 1.6% and March 2010 around 0.65%.

In [ ]:
MONTH_NUM = {m: i for i, m in enumerate(["jan","feb","mar","apr","may","jun","jul","aug","sep","oct","nov","dec"], 1)}
YEAR_BY_NR_EMPLOYED = {5191.0: 2008, 5228.1: 2008, 5195.8: 2008, 5176.3: 2008,
                       5099.1: 2009, 5076.2: 2009, 5017.5: 2009, 5023.5: 2009,
                       5008.7: 2010, 4991.6: 2010, 4963.6: 2010}

def _find_csv_in_zip(zbytes, name):
    z = zipfile.ZipFile(io.BytesIO(zbytes))
    for n in z.namelist():
        if n.endswith(name): return z.read(n)
        if n.endswith(".zip"):
            hit = _find_csv_in_zip(z.read(n), name)
            if hit is not None: return hit
    return None

def load_bank():
    try:
        raw = urllib.request.urlopen("https://archive.ics.uci.edu/static/public/222/bank+marketing.zip", timeout=60).read()
        df = pd.read_csv(io.BytesIO(_find_csv_in_zip(raw, "bank-additional-full.csv")), sep=";")
        src, ordered = "UCI bank-additional-full.csv (original, date-ordered)", True
    except Exception:
        url = "https://raw.githubusercontent.com/madmashup/targeted-marketing-predictive-engine/master/banking.csv"
        df = pd.read_csv(url); src, ordered = f"public mirror, rows shuffled ({url})", False
    df.columns = [c.replace(".", "_") for c in df.columns]
    df["y"] = df["y"].astype(str).isin(["yes", "1"]).astype(int)
    year = df["nr_employed"].round(1).map(YEAR_BY_NR_EMPLOYED)
    assert year.notna().all(), "unexpected nr_employed value"
    df["period"] = year.astype(int) * 100 + df["month"].map(MONTH_NUM)
    # Checks: 26 months; each monthly CPI value belongs to exactly one month; date order if file is original
    assert df["period"].nunique() == 26
    assert df.groupby(["month", "cons_price_idx"])["period"].nunique().max() == 1
    if ordered: assert df["period"].is_monotonic_increasing, "reconstructed timeline disagrees with file order"
    df = df.sort_values("period", kind="stable").reset_index(drop=True)
    return df, src

bank, src_b = load_bank()
print("Source:", src_b, "| shape", bank.shape, "| subscribe rate %.3f" % bank["y"].mean())
timeline = bank.groupby("period").agg(calls=("y", "size"), subscribe_rate=("y", "mean"), euribor3m=("euribor3m", "median"))
fig, ax = plt.subplots(1, 2, figsize=(12, 3.2))
lbl = [f"{p//100}-{p%100:02d}" for p in timeline.index]
ax[0].bar(lbl, timeline["calls"]); ax[0].set_title("Calls per month"); ax[0].tick_params(axis="x", rotation=90, labelsize=7)
ax[1].plot(lbl, timeline["subscribe_rate"], marker="o"); ax[1].set_title("Subscription rate per month"); ax[1].tick_params(axis="x", rotation=90, labelsize=7)
plt.tight_layout(); plt.savefig(f"{OUT}/bank_timeline.png", dpi=150); plt.show()
timeline.T.round(3)

**Look at the right-hand chart before reading on.** In 2008 the bank made thousands of calls a month and about 5% subscribed. By 2010 it made a few hundred calls a month and around half subscribed. A model trained on 2008–09 and tested on 2010 faces a different world, and a random split hides this completely.

### B1. Data leakage: the `duration` column
`duration` is how long the call lasted. It is strongly predictive, but **you only know it after the call ends**, and by then you already know the outcome. The dataset's own documentation says it should be dropped for any realistic model. This is a standard example of leakage: a feature that wouldn't exist at the moment you make the prediction.

In [ ]:
FEATS_ALL = [c for c in bank.columns if c not in ("y", "period")]
FEATS = [c for c in FEATS_ALL if c != "duration"]
Xb_all = as_model_frame(bank[FEATS_ALL]); yb = bank["y"]
idx_tr, idx_te = train_test_split(np.arange(len(bank)), test_size=0.2, stratify=yb, random_state=SEED)

b1_models = {"Logistic regression": run_logreg, "XGBoost (sensible defaults)": run_xgb, "TabPFN v2 (defaults)": run_tabpfn}
parts = []
for label, cols in [("WITH duration (leaky)", FEATS_ALL), ("WITHOUT duration (realistic)", FEATS)]:
    X = Xb_all[cols]
    df, _ = run_all(b1_models, X.iloc[idx_tr], yb.iloc[idx_tr], X.iloc[idx_te], yb.iloc[idx_te], tag=label)
    parts.append(df)
b1 = pd.concat(parts)
SUMMARY.append(("B1. Leakage: with vs without `duration` (random stratified split)", show(b1)))
show(b1)

Every model looks excellent when it can see `duration`, and every model gets worse without it. **Leakage helps all models, so a benchmark that includes it doesn't tell you which model is better. It tells you the benchmark is broken.** From here on, `duration` is excluded.

### B2. Random split vs. temporal split
* **Stratified random split:** rows from all 26 months are shuffled into train and test. The model is tested on months it has already seen.
* **Temporal split:** train on May 2008 – Dec 2009, test on Mar – Nov 2010. This is how the model would actually be used: trained on the past, applied to the future.

TabPFN v2 reads at most `MAX_CONTEXT` rows (10,000 on GPU), so it uses a stratified sample of the training set. To keep the comparison fair, we also train **XGBoost on the same sample** ("XGBoost @ TabPFN's N"). That separates "better model" from "more data".

In [ ]:
Xb = Xb_all[FEATS]
def xgb_same_n(Xtr, ytr, Xte): Xc, yc = context_sample(Xtr, ytr, MAX_CONTEXT); return run_xgb(Xc, yc, Xte)
b2_models = {
    "Prior (base rate)": run_prior, "Logistic regression": run_logreg,
    "XGBoost (sensible defaults)": run_xgb, "XGBoost (CV-tuned rounds)": run_xgb_cv, "LightGBM (defaults)": run_lgbm,
    "XGBoost @ TabPFN's N": xgb_same_n, "TabPFN v2 (defaults)": run_tabpfn,
}
tr_time = bank["period"] < 201001; te_time = bank["period"] >= 201001
splits = {
    "Random stratified": (idx_tr, idx_te),
    "Temporal (train 2008–09, test 2010)": (np.where(tr_time)[0], np.where(te_time)[0]),
}
b2_parts, b2_preds = [], {}
for s, (tr, te) in splits.items():
    df, preds = run_all(b2_models, Xb.iloc[tr], yb.iloc[tr], Xb.iloc[te], yb.iloc[te], tag=s)
    b2_parts.append(df); b2_preds[s] = (yb.iloc[te].values, preds)
    print(f"{s}: train n={len(tr):,} (rate {yb.iloc[tr].mean():.3f}) | test n={len(te):,} (rate {yb.iloc[te].mean():.3f})")
b2 = pd.concat(b2_parts)
SUMMARY.append(("B2. Random vs temporal split (no `duration`)", show(b2)))
show(b2)

**What to look for.**
* **ROC-AUC usually drops under the temporal split.** The random split was flattering every model.
* **Calibration falls apart.** Compare *Mean pred* with *Base rate* on the temporal split. 2010's real rate (about 52%) is far from anything in the training years, and the models miss it in both directions (in the reference run, XGBoost averaged 30%, logistic regression 71% and TabPFN 47%). Log loss and Brier score show the damage that AUC hides. AUC only measures ranking; it ignores whether the probabilities are right.
* **Rankings between models can change** from one split to the other. A model that "wins" on a random split hasn't necessarily won.

### B2b. Rolling-origin backtest: retrain each month on recent data
A more realistic setup: for each month of 2010, train only on the **three months before it** and predict that month. Training sets become small (hundreds to about 1,500 rows), which is where TabPFN is designed to be strongest. We also compare against training on **all** history.

In [ ]:
periods = sorted(bank["period"].unique()); test_periods = [p for p in periods if p >= 201001]
roll_models = {
    "Prior (base rate), last 3 months": ("recent", run_prior),
    "Logistic regression, last 3 months": ("recent", run_logreg),
    "XGBoost, last 3 months": ("recent", run_xgb),
    "LightGBM, last 3 months": ("recent", run_lgbm),
    "TabPFN v2, last 3 months": ("recent", run_tabpfn),
    "XGBoost, all history": ("all", run_xgb),
    "TabPFN v2, all history (most recent N rows)": ("all", lambda a, b, c: run_tabpfn(a, b, c, recent=True)),
}
roll_rows = []
for tp in test_periods:
    k = periods.index(tp); prev3 = periods[k-3:k]
    te = bank["period"] == tp
    for name, (window, fn) in roll_models.items():
        tr = bank["period"].isin(prev3) if window == "recent" else bank["period"] < tp
        p, f, pr = fn(Xb[tr], yb[tr], Xb[te])
        roll_rows.append({"test month": tp, "Model": name, "n_train": int(tr.sum()), "n_test": int(te.sum()), **metrics(yb[te], p), "Fit+predict s": f + pr})
roll = pd.DataFrame(roll_rows)
b2b = roll.groupby("Model").agg(**{"ROC-AUC (mean of 9 months)": ("ROC-AUC", "mean"), "ROC-AUC sd": ("ROC-AUC", "std"),
                                   "Log loss": ("Log loss", "mean"), "Brier": ("Brier", "mean"),
                                   "median n_train": ("n_train", "median"), "s per month": ("Fit+predict s", "mean")}).sort_values("ROC-AUC (mean of 9 months)", ascending=False)
wide_r = roll.pivot(index="test month", columns="Model", values="ROC-AUC")
print("TabPFN (last 3 months) beats XGBoost (last 3 months) in %d/%d months" % ((wide_r["TabPFN v2, last 3 months"] > wide_r["XGBoost, last 3 months"]).sum(), len(wide_r)))
SUMMARY.append(("B2b. Rolling-origin backtest over 2010 (9 monthly test sets)", b2b.round(4)))
b2b.round(4)

Each test month has only about 130–310 calls, so the per-month AUC is noisy (see the `sd` column). Judge by consistency across months and by log loss, not by the third decimal place. Two questions: does TabPFN beat the other models **on the same small windows**, and does a short recent window beat simply using all the history? (Reference run: on 3-month windows of about 750 rows, TabPFN averaged 0.738, against 0.706 for logistic regression and 0.682 for XGBoost on the same windows. It also beat XGBoost trained on all history, 0.714. This is a mean over 9 small months with no formal test, so treat it as supporting evidence, not proof.)

### B3. Class imbalance: which metrics still mean something
Only about 11% of calls end in a subscription. A model that **always says "no" is about 89% accurate** and completely useless, which is why we never report accuracy here.
* **ROC-AUC** measures ranking quality. It isn't affected by the base rate, but it can look good even when the positive class is rare and hard to find.
* **PR-AUC (average precision)** focuses on the positives. Its "random" level equals the base rate (about 0.11), not 0.5.
* **Precision and lift at the top 10%** answer the business question: *if we can only call 10% of customers, how many of those calls succeed?*

We **don't** oversample or reweight here. Those tricks change predicted probabilities and damage calibration. If you need a decision threshold, choose it from costs (price of a call vs. value of a subscription) using calibrated probabilities. (TabPFN has a `balance_probabilities` option for a different trade-off. We leave it off so the comparison stays like-for-like.)

In [ ]:
def top_k_stats(y, p, frac=0.10):
    y = np.asarray(y); n = max(1, int(len(y) * frac)); top = np.argsort(-np.asarray(p))[:n]
    prec = y[top].mean(); return prec, y[top].sum() / y.sum(), prec / y.mean()
rows = []
for s, (yt, preds) in b2_preds.items():
    for name in ["Prior (base rate)", "Logistic regression", "XGBoost (sensible defaults)", "TabPFN v2 (defaults)"]:
        prec, rec, lift = top_k_stats(yt, preds[name])
        rows.append({"Split": s, "Model": name, "Base rate": yt.mean(), "Accuracy of always-'no'": 1 - yt.mean(),
                     "PR-AUC": average_precision_score(yt, preds[name]), "Precision@top10%": prec, "Recall@top10%": rec, "Lift@top10%": lift})
b3 = pd.DataFrame(rows)
SUMMARY.append(("B3. Imbalance-aware metrics", b3.round(3)))
b3.round(3)

### B4. Inference cost: how time grows with training-set size
TabPFN does its "learning" at prediction time, so **prediction cost grows with the amount of context**, and it needs a GPU to be practical. Gradient-boosted trees take longer to train but predict almost instantly, whatever the training size. We vary the number of training rows and keep a fixed set of 2,000 test rows.

In [ ]:
pool_tr = idx_tr; te_fixed = idx_te[:2000]
sizes = [s for s in [250, 500, 1000, 2000, 5000, 10000] if s <= MAX_CONTEXT]
scale_models = {"Logistic regression": run_logreg, "XGBoost (sensible defaults)": run_xgb, "TabPFN v2 (defaults)": run_tabpfn}
sc_rows = []
for n in sizes:
    Xn, yn = context_sample(Xb.iloc[pool_tr], yb.iloc[pool_tr], n, seed=SEED)
    for name, fn in scale_models.items():
        p, f, pr = fn(Xn, yn, Xb.iloc[te_fixed])
        sc_rows.append({"n_train": n, "Model": name, "ROC-AUC": roc_auc_score(yb.iloc[te_fixed], p), "Fit s": f, "Predict s (2k rows)": pr, "ms per predicted row": 1000 * pr / len(te_fixed)})
b4 = pd.DataFrame(sc_rows)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for name, g in b4.groupby("Model"):
    ax[0].plot(g["n_train"], g["ROC-AUC"], marker="o", label=name)
    ax[1].plot(g["n_train"], g["Fit s"] + g["Predict s (2k rows)"], marker="o", label=name)
ax[0].set(xscale="log", xlabel="training rows", ylabel="ROC-AUC", title="Learning curve (random split, no duration)")
ax[1].set(xscale="log", yscale="log", xlabel="training rows", ylabel="fit + predict seconds", title=f"Cost ({'GPU' if HAS_GPU else 'CPU'})")
ax[0].legend(fontsize=8); plt.tight_layout(); plt.savefig(f"{OUT}/bank_scaling.png", dpi=150); plt.show()
SUMMARY.append((f"B4. Scaling with training size ({'GPU: ' + torch.cuda.get_device_name(0) if HAS_GPU else 'CPU'})", b4.round(4)))
b4.round(4)

**What to look for:** TabPFN's advantage, if it has one, is usually **largest at small n** and shrinks as boosted trees get more data. Each point is one random sample, so expect some wobble (in the reference run TabPFN dipped at 2,000 rows). Check the per-row prediction cost too. For batch scoring a few thousand rows that cost doesn't matter; for a real-time service handling millions of requests it might decide the question on its own. (Prior Labs sells an enterprise "fast inference" mode for that case; we only measure the open-source library here.)

### B5. Uncertainty: calibration, confidence intervals and seed variation
Three separate questions:
1. **Calibration:** when a model says 30%, does the event happen about 30% of the time? (Reliability curves and expected calibration error, ECE.)
2. **Statistical uncertainty:** is the gap between two models bigger than chance? (Paired bootstrap on the test set.)
3. **Run-to-run variation:** how much does the result move with a different random seed or a different context sample?

In [ ]:
def ece(y, p, bins=10):
    y = np.asarray(y); p = np.asarray(p); edges = np.linspace(0, 1, bins + 1); idx = np.clip(np.digitize(p, edges) - 1, 0, bins - 1)
    return sum((idx == b).mean() * abs(y[idx == b].mean() - p[idx == b].mean()) for b in range(bins) if (idx == b).any())

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5)); cal_rows = []
for ax, (s, (yt, preds)) in zip(axes, b2_preds.items()):
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    for name in ["Logistic regression", "XGBoost (sensible defaults)", "TabPFN v2 (defaults)"]:
        fy, fp = calibration_curve(yt, preds[name], n_bins=10, strategy="quantile")
        ax.plot(fp, fy, marker="o", label=name); cal_rows.append({"Split": s, "Model": name, "ECE": ece(yt, preds[name])})
    ax.set(title=s, xlabel="predicted probability", ylabel="observed frequency"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/bank_calibration.png", dpi=150); plt.show()

diff_rows = []
for s, (yt, preds) in b2_preds.items():
    for other in ["XGBoost (sensible defaults)", "XGBoost @ TabPFN's N", "Logistic regression"]:
        m, lo, hi, share = paired_boot_diff(yt, preds["TabPFN v2 (defaults)"], preds[other], B=500)
        diff_rows.append({"Split": s, "Comparison": f"TabPFN − {other}", "ΔAUC": m, "95% CI low": lo, "95% CI high": hi, "share of resamples TabPFN ahead": share})

seed_rows = []; te_sub = idx_te[:2000]
for seed in range(3):
    p, *_ = run_tabpfn(Xb.iloc[idx_tr], yb.iloc[idx_tr], Xb.iloc[te_sub], seed=seed)
    seed_rows.append({"Model": "TabPFN v2 (context sample + seed)", "seed": seed, "ROC-AUC": roc_auc_score(yb.iloc[te_sub], p)})
    p, *_ = run_xgb(Xb.iloc[idx_tr], yb.iloc[idx_tr], Xb.iloc[te_sub], seed=SEED + seed)
    seed_rows.append({"Model": "XGBoost (row/column subsampling seed)", "seed": seed, "ROC-AUC": roc_auc_score(yb.iloc[te_sub], p)})
seeds = pd.DataFrame(seed_rows).groupby("Model")["ROC-AUC"].agg(["mean", "std", "min", "max"])

b5a, b5b = pd.DataFrame(cal_rows), pd.DataFrame(diff_rows)
SUMMARY += [("B5a. Expected calibration error (lower is better)", b5a.round(4)),
            ("B5b. Paired bootstrap ΔAUC (TabPFN minus baseline)", b5b.round(4)),
            ("B5c. Seed sensitivity (2,000-row test subset, 3 seeds)", seeds.round(4))]
display(b5a.round(4)); display(b5b.round(4)); display(seeds.round(4))

---
# Summary
The cell below writes every table to `results/summary.md` and the charts to `results/*.png`, ready for a README or blog post.

### Where TabPFN helped, and where it didn't (reference run)
| Helped | Didn't help, or cost more |
|---|---|
| **Small tables:** +0.036 AUC over logistic regression at 250 rows (B4), and best on 3-month rolling windows (B2b) | **Large tables:** with 33k rows available, XGBoost beat TabPFN's 10k-row context by 0.010 AUC (B2, B5b). On equal data they tied |
| **Shift over time:** best AUC, log loss and calibration when tested on 2010 (B2, B5a) | **Prediction cost:** about 1,000× XGBoost per row at 10k context, and it needs a GPU (B4) |
| **No tuning, stable:** seed-to-seed sd of 0.001 AUC (B5c) | **Calibration on the random split:** predicted 9.3% vs a true 11.3% base rate (ECE 0.020 vs 0.010 for XGBoost) (B2, B5a) |
| | **German credit:** best on average, but within noise (A2, A3) |

**What this does *not* show:** why TabPFN held up better in 2010. This is one temporal split on one dataset, and the 2010 economic indicators fall outside the training range. A natural next experiment is to drop the macro indicators and re-test. Nothing here generalises beyond these two datasets.

In [ ]:
with open(f"{OUT}/summary.md", "w") as fh:
    fh.write(f"# Results\n\nRun on {'GPU: ' + torch.cuda.get_device_name(0) if HAS_GPU else 'CPU'} · tabpfn {tabpfn.__version__} (TabPFN-2 weights) · xgboost {xgb.__version__} · lightgbm {lgb.__version__} · TabPFN context cap {MAX_CONTEXT:,}\n\n")
    fh.write(f"German credit source: {src_g}  \nBank marketing source: {src_b}\n\n")
    if SMOKE_CKPT: fh.write("**SMOKE TEST: TabPFN rows use random weights and are meaningless.**\n\n")
    for title, df in SUMMARY:
        fh.write(f"## {title}\n\n{df.to_markdown()}\n\n")
print(open(f"{OUT}/summary.md").read()[:3000], "...")
print("\nFiles:", sorted(os.listdir(OUT)))